# HoloKai GLiNER — Sweep A: learning rate × epochs

**Owner:** person A. Four runs, everything identical to Run 1 except the learning rate, and **5 epochs** each:

| Run | learning_rate | epochs |
|---|---|---|
| A1 | 5e-6 | 5 |
| A2 | 1e-5 | 5 |
| A3 | 2e-5 | 5 |
| A4 | 3e-5 | 5 |

Every epoch is saved and evaluated, so each run gives epochs 1–5 at once → a learning-rate × epoch table.

**What it uses (already on Drive, nothing to upload except this notebook):** the shared fine-tuning folder
(`finetune` / `finetune1`, auto-detected in cell 2): `data/*_mapping.json`, `scripts/`, `configs/finetune_run_1.json`,
`outputs/finetune_run_1/baseline/`, `requirements-colab.txt`.

**What it saves to Drive:** everything goes into `Holokai_Project/finetune_A/` (created automatically): `configs/` (the four configs), `outputs/<run>/` — metrics, per-label CSVs, predictions, training
history, threshold tuning, V4 test, flat_ner check, run summary — and `sweep_A_table.csv`. Nothing is written into the shared finetune folder.
Model weights stay on the temporary Colab disk and are deleted after each run (they would fill Drive).

**Steps:** Runtime → Change runtime type → **T4 GPU** → Connect. Then run the cells top to bottom.
If the session disconnects, reconnect, rerun cells 1–4, then continue from the next unfinished run.

## 1. Check the GPU

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
assert torch.cuda.is_available(), "Enable a T4 GPU runtime first (Runtime -> Change runtime type)."

## 2. Mount Drive and find the project

`Holokai_Project` is a **shared** folder. If the check below fails, open Google Drive → "Shared with me" →
right-click **Holokai_Project** → **Organize → Add shortcut** → choose **My Drive**, then rerun this cell.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
PROJECT = Path("/content/drive/MyDrive/Holokai_Project")
FT_NAME = None   # leave None to auto-detect; or set e.g. "finetune1"
REQUIRED = ["data/train_gliner_mapping.json", "data/val_gliner_mapping.json", "data/corpus_V4_gliner_mapping.json",
            "scripts/train_gliner.py", "configs/finetune_run_1.json", "outputs/finetune_run_1/baseline/metrics.json",
            "requirements-colab.txt"]
assert PROJECT.exists(), "Holokai_Project not found in MyDrive (see the shortcut note above)."
A_DIR = PROJECT / "finetune_A"   # everything this sweep creates goes here
names = [FT_NAME] if FT_NAME else sorted(p.name for p in PROJECT.iterdir()
                                         if p.is_dir() and p.name.startswith("finetune") and p != A_DIR)
for name in names:
    missing = [r for r in REQUIRED if not (PROJECT / name / r).exists()]
    print(f"{name}: " + ("all files found" if not missing else "missing " + ", ".join(missing)))
    if not missing:
        FT = PROJECT / name
        break
else:
    raise FileNotFoundError("No finetune folder has all required files (see list above).")
print("Reading shared inputs from", FT)
print("Writing all sweep A files to", A_DIR)
%cd "$FT"

## 3. Install the pinned libraries (same as Run 1)

In [ ]:
!pip -q install -r requirements-colab.txt

## 4. Load the pipeline and create the four configs

In [ ]:
PIPELINE_SOURCE = r'''
import csv, json, os, shutil, subprocess, sys, time
from pathlib import Path

PROVIDER = "name medical professional"
BASE_MODEL = "knowledgator/gliner-pii-base-v1.0"


def _run(args, log_path):
    """Run a finetune script, stream its output to the notebook and to a log file."""
    print(">>", " ".join(map(str, args)), flush=True)
    t0 = time.time()
    with open(log_path, "a", encoding="utf-8") as log:
        proc = subprocess.Popen([sys.executable, *map(str, args)], stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
        for line in proc.stdout:
            log.write(line)
            if "Evaluated" not in line and "Fetching" not in line:
                print(line, end="", flush=True)
        code = proc.wait()
    print(f"   exit {code} after {(time.time() - t0) / 60:.1f} min", flush=True)
    return code


def _per_label(d):
    with open(Path(d) / "per_label_metrics.csv", encoding="utf-8") as f:
        return {r["label"]: r for r in csv.DictReader(f)}


def _load(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def make_config(base_config, name, lr, epochs, train_data, val_data, work_root, config_dir, device="auto"):
    cfg = json.loads(Path(base_config).read_text(encoding="utf-8"))
    cfg.update({"train_data": str(train_data), "val_data": str(val_data),
                "output_dir": str(Path(work_root) / name / "training"),
                "learning_rate": lr, "epochs": epochs, "device": device})
    path = Path(config_dir) / f"{name}.json"
    path.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    return path


def run_one(name, config_path, scripts, val_data, v4_data, baseline_dir, work_root, drive_out,
            device="auto", keep_best_weights=False, tune_thresholds=True):
    """Train one config, evaluate every epoch, select, compare, tune, test on V4, save small files."""
    scripts, work, drive = Path(scripts), Path(work_root) / name, Path(drive_out) / name
    train_out, eval_out = work / "training", work / "model_eval"
    drive.mkdir(parents=True, exist_ok=True)
    log = drive / "pipeline_log.txt"
    ev = lambda model, data, out, flat=False: _run(
        [scripts / "evaluate_gliner.py", "--label-schema", "recommended", "--model", model, "--data", data,
         "--threshold", "0.4", "--output-dir", out, "--device", device] + (["--flat-ner"] if flat else []), log)

    if not (train_out / "final_model").exists():
        if _run([scripts / "train_gliner.py", "--config", config_path], log) != 0:
            raise RuntimeError(f"training failed for {name}; see {log}")

    models = sorted([p.name for p in train_out.iterdir() if p.is_dir() and (p.name.startswith("epoch-") or p.name == "final_model")
                     and any(p.glob("*.safetensors"))],
                    key=lambda x: (x == "final_model", int(x.split("-")[1]) if x.startswith("epoch-") else 0))
    rows = []
    for m in models:
        if not (eval_out / m / "metrics.json").exists():
            ev(train_out / m, val_data, eval_out / m)
        o = _load(eval_out / m / "metrics.json")["overall"]
        rows.append({"model": m, "precision": o["precision"], "recall": o["recall"], "f1": o["f1"],
                     "fp": o["fp"], "fn": o["fn"], "provider_f1": float(_per_label(eval_out / m)[PROVIDER]["f1"])})
    ranked = sorted(rows, key=lambda r: (r["f1"], r["provider_f1"], r["precision"]), reverse=True)
    best = ranked[0]["model"]
    best_path = train_out / best
    print(f"\nSelected {best}: F1 {ranked[0]['f1']:.4f}", flush=True)

    base, ft = _load(Path(baseline_dir) / "metrics.json")["overall"], _load(eval_out / best / "metrics.json")["overall"]
    (work / "comparison").mkdir(parents=True, exist_ok=True)
    (work / "comparison" / "pretrained_vs_finetuned_overall.json").write_text(json.dumps(
        {"threshold": 0.4, "pretrained": base, "fine_tuned": ft,
         "delta": {k: ft[k] - base[k] for k in ("precision", "recall", "f1")}}, indent=2), encoding="utf-8")

    calibrated = None
    if tune_thresholds:
        if _run([scripts / "tune_per_label_thresholds.py", "--label-schema", "recommended", "--model", best_path,
                 "--data", val_data, "--output-dir", work / "threshold_tuning", "--device", device], log) == 0:
            calibrated = _load(work / "threshold_tuning" / "calibrated_metrics.json")["overall"]
        else:
            print("threshold tuning failed (e.g. provider recall < 0.70); continuing", flush=True)

    ev(best_path, val_data, work / "extra" / "val_flat_ner", flat=True)
    ev(best_path, v4_data, work / "extra" / "v4_finetuned")

    cfg = _load(train_out / "training_config.json")
    summary = {"run": name, "learning_rate": cfg["learning_rate"], "epochs": cfg["epochs"],
               "batch_size": cfg["batch_size"], "seed": cfg["seed"], "selected_model": best,
               "epochs_table": rows, "finetuned_overall_at_0_4": ft, "pretrained_overall_at_0_4": base,
               "calibrated_validation_overall": calibrated,
               "val_flat_ner_overall": _load(work / "extra/val_flat_ner/metrics.json")["overall"],
               "v4_finetuned_overall": _load(work / "extra/v4_finetuned/metrics.json")["overall"],
               "data_manifest": _load(train_out / "data_manifest.json")}
    (work / "run_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    # copy everything except model weights to Drive
    for sub in ("model_eval", "comparison", "threshold_tuning", "extra"):
        if (work / sub).exists():
            shutil.copytree(work / sub, drive / sub, dirs_exist_ok=True)
    (drive / "training").mkdir(exist_ok=True)
    for f in train_out.glob("*.json"):
        shutil.copy2(f, drive / "training" / f.name)
    shutil.copy2(work / "run_summary.json", drive / "run_summary.json")
    shutil.copy2(config_path, drive / Path(config_path).name)

    # free local disk: drop all weights (optionally keep the selected model)
    for m in models:
        if keep_best_weights and m == best:
            continue
        shutil.rmtree(train_out / m, ignore_errors=True)
    print(f"Saved results for {name} to {drive}", flush=True)
    return summary


def build_table(drive_out, names, path_csv):
    """Learning rate x epoch table of validation F1 (threshold 0.4), plus best/V4 per run."""
    lines = []
    for n in names:
        p = Path(drive_out) / n / "run_summary.json"
        if not p.exists():
            continue
        s = _load(p)
        row = {"run": n, "learning_rate": s["learning_rate"], "epochs": s["epochs"]}
        for r in s["epochs_table"]:
            row[f"{r['model']}_f1"] = round(100 * r["f1"], 2)
        row.update({"selected": s["selected_model"], "best_val_f1": round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
                    "best_val_precision": round(100 * s["finetuned_overall_at_0_4"]["precision"], 2),
                    "best_val_recall": round(100 * s["finetuned_overall_at_0_4"]["recall"], 2),
                    "flat_ner_val_f1": round(100 * s["val_flat_ner_overall"]["f1"], 2),
                    "calibrated_val_f1": round(100 * s["calibrated_validation_overall"]["f1"], 2) if s["calibrated_validation_overall"] else None,
                    "v4_f1": round(100 * s["v4_finetuned_overall"]["f1"], 2)})
        lines.append(row)
    keys = []
    for r in lines:
        keys += [k for k in r if k not in keys]
    with open(path_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(lines)
    return lines
'''
exec(PIPELINE_SOURCE)

In [ ]:
import os
SCRIPTS = FT / "scripts"
TRAIN, VAL, V4 = FT / "data/train_gliner_mapping.json", FT / "data/val_gliner_mapping.json", FT / "data/corpus_V4_gliner_mapping.json"
BASELINE = FT / "outputs/finetune_run_1/baseline"
WORK = Path("/content/sweep_A_work")          # temporary Colab disk (model weights live here)
DRIVE_OUT = A_DIR / "outputs"                 # small result files go to Drive
CONFIG_DIR = A_DIR / "configs"
for d in (WORK, DRIVE_OUT, CONFIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

RUNS = {"A1_lr5e-6_ep5": 5e-6, "A2_lr1e-5_ep5": 1e-5, "A3_lr2e-5_ep5": 2e-5, "A4_lr3e-5_ep5": 3e-5}
CONFIGS = {name: make_config(FT / "configs/finetune_run_1.json", f"finetune_sweep_{name}", lr, 5, TRAIN, VAL, WORK, CONFIG_DIR)
           for name, lr in RUNS.items()}
for name, path in CONFIGS.items():
    print(name, "->", path)
    !python scripts/train_gliner.py --config "{path}" --check-config | tail -1

## 5. Run A1 — learning rate 5e-6, 5 epochs (about 10–15 min)

In [ ]:
s = run_one("finetune_sweep_A1_lr5e-6_ep5", CONFIGS["A1_lr5e-6_ep5"], SCRIPTS, VAL, V4, BASELINE, WORK, DRIVE_OUT)
print("best:", s["selected_model"], "| val F1 @0.4:", round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
      "| V4 F1:", round(100 * s["v4_finetuned_overall"]["f1"], 2))

## 6. Run A2 — learning rate 1e-5, 5 epochs (about 10–15 min)

In [ ]:
s = run_one("finetune_sweep_A2_lr1e-5_ep5", CONFIGS["A2_lr1e-5_ep5"], SCRIPTS, VAL, V4, BASELINE, WORK, DRIVE_OUT)
print("best:", s["selected_model"], "| val F1 @0.4:", round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
      "| V4 F1:", round(100 * s["v4_finetuned_overall"]["f1"], 2))

## 7. Run A3 — learning rate 2e-5, 5 epochs (about 10–15 min)

In [ ]:
s = run_one("finetune_sweep_A3_lr2e-5_ep5", CONFIGS["A3_lr2e-5_ep5"], SCRIPTS, VAL, V4, BASELINE, WORK, DRIVE_OUT)
print("best:", s["selected_model"], "| val F1 @0.4:", round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
      "| V4 F1:", round(100 * s["v4_finetuned_overall"]["f1"], 2))

## 8. Run A4 — learning rate 3e-5, 5 epochs (about 10–15 min)

In [ ]:
s = run_one("finetune_sweep_A4_lr3e-5_ep5", CONFIGS["A4_lr3e-5_ep5"], SCRIPTS, VAL, V4, BASELINE, WORK, DRIVE_OUT)
print("best:", s["selected_model"], "| val F1 @0.4:", round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
      "| V4 F1:", round(100 * s["v4_finetuned_overall"]["f1"], 2))

## 9. Run A5 — learning rate 4e-5, 5 epochs (about 10–15 min)

In [ ]:
name, lr = "A5_lr4e-5_ep5", 4e-5
RUNS[name] = lr
run_name = f"finetune_sweep_{name}"
if (DRIVE_OUT / run_name / "run_summary.json").exists():
    print(name, "already done, skipping")
else:
    cfg = make_config(FT / "configs/finetune_run_1.json", run_name, lr, 5, TRAIN, VAL, WORK, CONFIG_DIR)
    s = run_one(run_name, cfg, SCRIPTS, VAL, V4, BASELINE, WORK, DRIVE_OUT)
    print("best:", s["selected_model"], "| val F1 @0.4:", round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
          "| V4 F1:", round(100 * s["v4_finetuned_overall"]["f1"], 2))

## 10. Run A6 — learning rate 5e-5, 5 epochs (about 10–15 min)

In [ ]:
name, lr = "A6_lr5e-5_ep5", 5e-5
RUNS[name] = lr
run_name = f"finetune_sweep_{name}"
if (DRIVE_OUT / run_name / "run_summary.json").exists():
    print(name, "already done, skipping")
else:
    cfg = make_config(FT / "configs/finetune_run_1.json", run_name, lr, 5, TRAIN, VAL, WORK, CONFIG_DIR)
    s = run_one(run_name, cfg, SCRIPTS, VAL, V4, BASELINE, WORK, DRIVE_OUT)
    print("best:", s["selected_model"], "| val F1 @0.4:", round(100 * s["finetuned_overall_at_0_4"]["f1"], 2),
          "| V4 F1:", round(100 * s["v4_finetuned_overall"]["f1"], 2))

## 11. Learning rate × epoch table (validation F1 at threshold 0.4)

In [ ]:
import pandas as pd
names = [f"finetune_sweep_{n}" for n in RUNS]
build_table(DRIVE_OUT, names, A_DIR / "sweep_A_table.csv")
display(pd.read_csv(A_DIR / "sweep_A_table.csv"))
print("Saved:", A_DIR / "sweep_A_table.csv")